In [ ]:
# Cell 1 - Setup
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import tensorflow.keras.backend as K
from PIL import Image
from sklearn.model_selection import train_test_split

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TF version: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# Cell 2 - Unzip dataset to session storage
import zipfile

zip_path = '/content/drive/MyDrive/LungTumour/dataset.zip'  # change if different name
extract_path = '/content/dataset'

print("Unzipping... please wait")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Done! Files:")
for f in os.listdir(extract_path):
    print(" ", f)

Unzipping... please wait
Done! Files:
  LIDC-IDRI-slices


In [ ]:
# Cell 4 - Build X_train, y_train from raw PNG structure
DATASET_PATH = '/content/dataset/LIDC-IDRI-slices'
IMG_SIZE     = (128, 128)   # resize all slices to this
DEPTH        = 8            # slices per volume (pad/truncate to this)

def load_volume(folder, size=IMG_SIZE, depth=DEPTH):
    """Load all slices from a folder, sort by slice number, pad to fixed depth."""
    files = sorted(os.listdir(folder),
                   key=lambda x: int(x.replace('slice-','').replace('.png','')))
    slices = []
    for f in files:
        if f.endswith('.png'):
            img = Image.open(os.path.join(folder, f)).convert('L')
            img = img.resize(size, Image.BILINEAR)
            slices.append(np.array(img, dtype=np.float32))

    # Pad or truncate to fixed depth
    if len(slices) < depth:
        while len(slices) < depth:
            slices.append(np.zeros(size, dtype=np.float32))
    else:
        slices = slices[:depth]

    return np.stack(slices, axis=-1)  # shape: (128, 128, 8)

def load_consensus_mask(nodule_path, size=IMG_SIZE, depth=DEPTH, threshold=2):
    """Average 4 radiologist masks. Pixel = tumour if >= threshold radiologists agree."""
    all_masks = []
    for mask_id in ['mask-0','mask-1','mask-2','mask-3']:
        mask_folder = os.path.join(nodule_path, mask_id)
        if os.path.exists(mask_folder):
            m = load_volume(mask_folder, size, depth)
            all_masks.append(m)

    if len(all_masks) == 0:
        return np.zeros((*size, depth), dtype=np.float32)

    avg_mask = np.mean(all_masks, axis=0)
    consensus = (avg_mask > 127 * (threshold / 4)).astype(np.float32)
    return consensus

print("Loading dataset... this may take 2-3 minutes")

X_list, y_list = [], []
patients = sorted(os.listdir(DATASET_PATH))

for patient in patients:
    patient_path = os.path.join(DATASET_PATH, patient)
    if not os.path.isdir(patient_path):
        continue

    nodules = sorted(os.listdir(patient_path))
    for nodule in nodules:
        nodule_path = os.path.join(patient_path, nodule)
        img_folder  = os.path.join(nodule_path, 'images')

        if not os.path.exists(img_folder):
            continue

        try:
            image   = load_volume(img_folder)           # (128, 128, 8)
            mask    = load_consensus_mask(nodule_path)   # (128, 128, 8)

            # Normalize image to [0, 1]
            image = image / 255.0
            mask  = mask  / 255.0 if mask.max() > 1 else mask

            X_list.append(image)
            y_list.append(mask)
        except Exception as e:
            print(f"  Skipping {patient}/{nodule}: {e}")

X = np.array(X_list)[..., np.newaxis]  # (N, 128, 128, 8, 1)
y = np.array(y_list)[..., np.newaxis]  # (N, 128, 128, 8, 1)

print(f"\nTotal samples : {len(X)}")
print(f"X shape       : {X.shape}")
print(f"y shape       : {y.shape}")
print(f"X range       : {X.min():.3f} to {X.max():.3f}")
print(f"y unique vals : {np.unique(y)}")
print(f"Foreground %  : {100*y.mean():.2f}%")

Loading dataset... this may take 2-3 minutes

Total samples : 2630
X shape       : (2630, 128, 128, 8, 1)
y shape       : (2630, 128, 128, 8, 1)
X range       : 0.000 to 1.000
y unique vals : [0. 1.]
Foreground %  : 0.43%


In [ ]:
# Cell 5 - Loss functions
def bce_dice_loss(y_true, y_pred):
    bce   = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    d_loss = dice_loss(y_true, y_pred)
    return 0.5 * bce + 0.5 * d_loss

print("Loss functions defined successfully")

Loss functions defined successfully


In [ ]:
 #Load M3, freeze encoder, compile as M4 directly

# ──  Load M3 directly (no cloning) ────────────────────────
m4_model = load_model(
    '/content/drive/MyDrive/LungTumour/lung_segmentation_augmented.h5',
    custom_objects={
        'dice_coefficient': dice_coefficient,
        'dice_loss'       : dice_loss
    },
    compile=False
)
print("M3 loaded as M4 base")

# ── 3. Freeze encoder + bottleneck (layers 0-20) ────────────
for i, layer in enumerate(m4_model.layers):
    layer.trainable = True if i >= 21 else False

trainable = sum([tf.size(w).numpy() for w in m4_model.trainable_weights])
frozen    = sum([tf.size(w).numpy() for w in m4_model.non_trainable_weights])
print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {frozen:,}")

# ── 4. Compile ───────────────────────────────────────────────
m4_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=bce_dice_loss,
    metrics=[dice_coefficient]
)
print("Compiled successfully with BCE+Dice loss")

M3 loaded as M4 base
Trainable params : 553,569
Frozen params    : 860,576
Compiled successfully with BCE+Dice loss


In [ ]:
# New Cell 7 - Load from Drive + fix shape
import numpy as np

X_train = np.load('/content/drive/MyDrive/LungTumour/X_train.npy')
y_train = np.load('/content/drive/MyDrive/LungTumour/y_train.npy')
X_val   = np.load('/content/drive/MyDrive/LungTumour/X_val.npy')
y_val   = np.load('/content/drive/MyDrive/LungTumour/y_val.npy')

# Transpose to match M3: (N, 8, 128, 128, 1)
X_train = np.transpose(X_train, (0, 3, 1, 2, 4))
X_val   = np.transpose(X_val,   (0, 3, 1, 2, 4))
y_train = np.transpose(y_train, (0, 3, 1, 2, 4))
y_val   = np.transpose(y_val,   (0, 3, 1, 2, 4))

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val  :", X_val.shape)
print("y_val  :", y_val.shape)
assert X_train.shape[1:] == (8, 128, 128, 1), "Shape mismatch!"
print("Shape OK")

X_train: (2104, 8, 128, 128, 1)
y_train: (2104, 8, 128, 128, 1)
X_val  : (526, 8, 128, 128, 1)
y_val  : (526, 8, 128, 128, 1)
Shape OK


In [ ]:
#Callbacks + Training
from tensorflow.keras.callbacks import (ModelCheckpoint, EarlyStopping,
                                        ReduceLROnPlateau, CSVLogger)

callbacks = [
    ModelCheckpoint(
        filepath='/content/drive/MyDrive/LungTumour/M4_best.h5',
        monitor='val_dice_coefficient',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_dice_coefficient',
        mode='max',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_dice_coefficient',
        mode='max',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1
    ),
    CSVLogger(
        '/content/drive/MyDrive/LungTumour/M4_training_log.csv',
        append=True
    )
]

print("Starting training...")
history = m4_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=2,
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)

print(f"\nBest val Dice: {max(history.history['val_dice_coefficient']):.4f}")

Starting training...
Epoch 1/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step - dice_coefficient: 0.7401 - loss: 0.1370
Epoch 1: val_dice_coefficient improved from None to 0.71491, saving model to /content/drive/MyDrive/LungTumour/M4_best.h5



Epoch 1: finished saving model to /content/drive/MyDrive/LungTumour/M4_best.h5
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 261s 229ms/step - dice_coefficient: 0.7358 - loss: 0.1392 - val_dice_coefficient: 0.7149 - val_loss: 0.1501 - learning_rate: 1.0000e-04
Epoch 2/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - dice_coefficient: 0.7421 - loss: 0.1357
Epoch 2: val_dice_coefficient improved from 0.71491 to 0.73033, saving model to /content/drive/MyDrive/LungTumour/M4_best.h5



Epoch 2: finished saving model to /content/drive/MyDrive/LungTumour/M4_best.h5
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 240s 228ms/step - dice_coefficient: 0.7405 - loss: 0.1364 - val_dice_coefficient: 0.7303 - val_loss: 0.1423 - learning_rate: 1.0000e-04
Epoch 3/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - dice_coefficient: 0.7366 - loss: 0.1384
Epoch 3: val_dice_coefficient did not improve from 0.73033
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 239s 227ms/step - dice_coefficient: 0.7421 - loss: 0.1355 - val_dice_coefficient: 0.7301 - val_loss: 0.1423 - learning_rate: 1.0000e-04
Epoch 4/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - dice_coefficient: 0.7490 - loss: 0.1320
Epoch 4: val_dice_coefficient improved from 0.73033 to 0.73607, saving model to /content/drive/MyDrive/LungTumour/M4_best.h5



Epoch 4: finished saving model to /content/drive/MyDrive/LungTumour/M4_best.h5
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 239s 228ms/step - dice_coefficient: 0.7479 - loss: 0.1323 - val_dice_coefficient: 0.7361 - val_loss: 0.1391 - learning_rate: 1.0000e-04
Epoch 5/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - dice_coefficient: 0.7491 - loss: 0.1313
Epoch 5: val_dice_coefficient did not improve from 0.73607
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 238s 227ms/step - dice_coefficient: 0.7494 - loss: 0.1314 - val_dice_coefficient: 0.7350 - val_loss: 0.1389 - learning_rate: 1.0000e-04
Epoch 6/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - dice_coefficient: 0.7521 - loss: 0.1300
Epoch 6: val_dice_coefficient improved from 0.73607 to 0.73821, saving model to /content/drive/MyDrive/LungTumour/M4_best.h5



Epoch 6: finished saving model to /content/drive/MyDrive/LungTumour/M4_best.h5
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 239s 227ms/step - dice_coefficient: 0.7529 - loss: 0.1295 - val_dice_coefficient: 0.7382 - val_loss: 0.1383 - learning_rate: 1.0000e-04
Epoch 7/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - dice_coefficient: 0.7555 - loss: 0.1281
Epoch 7: val_dice_coefficient improved from 0.73821 to 0.73889, saving model to /content/drive/MyDrive/LungTumour/M4_best.h5



Epoch 7: finished saving model to /content/drive/MyDrive/LungTumour/M4_best.h5
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 239s 227ms/step - dice_coefficient: 0.7529 - loss: 0.1294 - val_dice_coefficient: 0.7389 - val_loss: 0.1379 - learning_rate: 1.0000e-04
Epoch 8/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - dice_coefficient: 0.7639 - loss: 0.1239
Epoch 8: val_dice_coefficient improved from 0.73889 to 0.74149, saving model to /content/drive/MyDrive/LungTumour/M4_best.h5



Epoch 8: finished saving model to /content/drive/MyDrive/LungTumour/M4_best.h5
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 239s 227ms/step - dice_coefficient: 0.7591 - loss: 0.1261 - val_dice_coefficient: 0.7415 - val_loss: 0.1365 - learning_rate: 1.0000e-04
Epoch 9/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - dice_coefficient: 0.7702 - loss: 0.1203
Epoch 9: val_dice_coefficient did not improve from 0.74149
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 239s 227ms/step - dice_coefficient: 0.7652 - loss: 0.1229 - val_dice_coefficient: 0.7173 - val_loss: 0.1480 - learning_rate: 1.0000e-04
Epoch 10/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step - dice_coefficient: 0.7718 - loss: 0.1197
Epoch 10: val_dice_coefficient did not improve from 0.74149
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 239s 228ms/step - dice_coefficient: 0.7640 - loss: 0.1236 - val_dice_coefficient: 0.7393 - val_loss: 0.1374 - learning_rate: 1.0000e-04
Epoch 11/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step - dice_coefficient: 0.7653 - loss: 0.1227
Epoc


Epoch 11: finished saving model to /content/drive/MyDrive/LungTumour/M4_best.h5
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 240s 228ms/step - dice_coefficient: 0.7657 - loss: 0.1227 - val_dice_coefficient: 0.7441 - val_loss: 0.1349 - learning_rate: 1.0000e-04
Epoch 12/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - dice_coefficient: 0.7693 - loss: 0.1207
Epoch 12: val_dice_coefficient did not improve from 0.74412
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 238s 227ms/step - dice_coefficient: 0.7721 - loss: 0.1193 - val_dice_coefficient: 0.7314 - val_loss: 0.1414 - learning_rate: 1.0000e-04
Epoch 13/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - dice_coefficient: 0.7659 - loss: 0.1224
Epoch 13: val_dice_coefficient improved from 0.74412 to 0.74412, saving model to /content/drive/MyDrive/LungTumour/M4_best.h5



Epoch 13: finished saving model to /content/drive/MyDrive/LungTumour/M4_best.h5
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 236s 224ms/step - dice_coefficient: 0.7708 - loss: 0.1199 - val_dice_coefficient: 0.7441 - val_loss: 0.1346 - learning_rate: 1.0000e-04
Epoch 14/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step - dice_coefficient: 0.7749 - loss: 0.1175
Epoch 14: val_dice_coefficient did not improve from 0.74412
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 234s 223ms/step - dice_coefficient: 0.7718 - loss: 0.1194 - val_dice_coefficient: 0.7359 - val_loss: 0.1388 - learning_rate: 1.0000e-04
Epoch 15/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step - dice_coefficient: 0.7705 - loss: 0.1203
Epoch 15: val_dice_coefficient did not improve from 0.74412

Epoch 15: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 234s 222ms/step - dice_coefficient: 0.7735 - loss: 0.1186 - val_dice_coefficient: 0.7370 - val_loss: 0.1394 - learning_rate: 1.0000e-04
Restoring model weight

In [ ]:
# Load best model (Phase 1 — val Dice 0.7441)
m4_model = load_model(
    '/content/drive/MyDrive/LungTumour/M4_best.h5',
    custom_objects={
        'dice_coefficient': dice_coefficient,
        'dice_loss'       : dice_loss,
        'bce_dice_loss'   : bce_dice_loss
    },
    compile=False
)
print("M4 Phase 1 loaded — val Dice 0.7441")

# Load data
X_train = np.load('/content/drive/MyDrive/LungTumour/X_train.npy')
y_train = np.load('/content/drive/MyDrive/LungTumour/y_train.npy')
X_val   = np.load('/content/drive/MyDrive/LungTumour/X_val.npy')
y_val   = np.load('/content/drive/MyDrive/LungTumour/y_val.npy')

X_train = np.transpose(X_train, (0, 3, 1, 2, 4))
X_val   = np.transpose(X_val,   (0, 3, 1, 2, 4))
y_train = np.transpose(y_train, (0, 3, 1, 2, 4))
y_val   = np.transpose(y_val,   (0, 3, 1, 2, 4))

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)

M4 Phase 1 loaded — val Dice 0.7441
X_train: (2104, 8, 128, 128, 1)
X_val  : (526, 8, 128, 128, 1)


In [ ]:
# Smarter unfreeze — only decoder (not full model)
# This prevents overfitting without needing more data

UNFREEZE_FROM = 28  # only unfreeze last 10 layers (28-37)

for i, layer in enumerate(m4_model.layers):
    layer.trainable = True if i >= UNFREEZE_FROM else False

trainable = sum([tf.size(w).numpy() for w in m4_model.trainable_weights])
frozen    = sum([tf.size(w).numpy() for w in m4_model.non_trainable_weights])
print(f"Trainable: {trainable:,}")
print(f"Frozen   : {frozen:,}")

# Compile with even lower LR
m4_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=2e-5,
        weight_decay=1e-5          # L2 regularization to fight overfitting
    ),
    loss=bce_dice_loss,
    metrics=[dice_coefficient]
)
print("Compiled — LR=2e-5 with L2 weight decay")

Trainable: 110,817
Frozen   : 1,303,328
Compiled — LR=2e-5 with L2 weight decay


In [ ]:
callbacks_p2 = [
    ModelCheckpoint(
        filepath='/content/drive/MyDrive/LungTumour/M4_phase2_best.h5',
        monitor='val_dice_coefficient',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_dice_coefficient',
        mode='max',
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_dice_coefficient',
        mode='max',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    CSVLogger(
        '/content/drive/MyDrive/LungTumour/M4_phase2_retry_log.csv',
        append=True
    )
]

print("Starting Phase 2 retry...")
print("Unfrozen: layers 28-37 only (last 10)")
print("="*50)

history = m4_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=2,
    callbacks=callbacks_p2,
    shuffle=True,
    verbose=1
)

best = max(history.history['val_dice_coefficient'])
print(f"\nPhase 2 retry best val Dice : {best:.4f}")
print(f"Phase 1 best val Dice       : 0.7441")
print(f"Better than Phase 1         : {best > 0.7441}")

Starting Phase 2 retry...
Unfrozen: layers 28-37 only (last 10)
Epoch 1/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - dice_coefficient: 0.7795 - loss: 0.1163
Epoch 1: val_dice_coefficient improved from None to 0.74531, saving model to /content/drive/MyDrive/LungTumour/M4_phase2_best.h5



Epoch 1: finished saving model to /content/drive/MyDrive/LungTumour/M4_phase2_best.h5
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 191s 167ms/step - dice_coefficient: 0.7801 - loss: 0.1153 - val_dice_coefficient: 0.7453 - val_loss: 0.1339 - learning_rate: 2.0000e-05
Epoch 2/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - dice_coefficient: 0.7857 - loss: 0.1123
Epoch 2: val_dice_coefficient did not improve from 0.74531
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 179s 170ms/step - dice_coefficient: 0.7820 - loss: 0.1141 - val_dice_coefficient: 0.7439 - val_loss: 0.1345 - learning_rate: 2.0000e-05
Epoch 3/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - dice_coefficient: 0.7923 - loss: 0.1084
Epoch 3: val_dice_coefficient improved from 0.74531 to 0.74654, saving model to /content/drive/MyDrive/LungTumour/M4_phase2_best.h5



Epoch 3: finished saving model to /content/drive/MyDrive/LungTumour/M4_phase2_best.h5
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 178s 170ms/step - dice_coefficient: 0.7854 - loss: 0.1124 - val_dice_coefficient: 0.7465 - val_loss: 0.1332 - learning_rate: 2.0000e-05
Epoch 4/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - dice_coefficient: 0.7891 - loss: 0.1108
Epoch 4: val_dice_coefficient did not improve from 0.74654
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 177s 168ms/step - dice_coefficient: 0.7851 - loss: 0.1125 - val_dice_coefficient: 0.7427 - val_loss: 0.1350 - learning_rate: 2.0000e-05
Epoch 5/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - dice_coefficient: 0.7823 - loss: 0.1145
Epoch 5: val_dice_coefficient improved from 0.74654 to 0.74759, saving model to /content/drive/MyDrive/LungTumour/M4_phase2_best.h5



Epoch 5: finished saving model to /content/drive/MyDrive/LungTumour/M4_phase2_best.h5
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 178s 169ms/step - dice_coefficient: 0.7835 - loss: 0.1132 - val_dice_coefficient: 0.7476 - val_loss: 0.1328 - learning_rate: 2.0000e-05
Epoch 6/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - dice_coefficient: 0.7868 - loss: 0.1113
Epoch 6: val_dice_coefficient did not improve from 0.74759
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 177s 169ms/step - dice_coefficient: 0.7849 - loss: 0.1126 - val_dice_coefficient: 0.7439 - val_loss: 0.1344 - learning_rate: 2.0000e-05
Epoch 7/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - dice_coefficient: 0.7877 - loss: 0.1107
Epoch 7: val_dice_coefficient improved from 0.74759 to 0.74836, saving model to /content/drive/MyDrive/LungTumour/M4_phase2_best.h5



Epoch 7: finished saving model to /content/drive/MyDrive/LungTumour/M4_phase2_best.h5
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 178s 169ms/step - dice_coefficient: 0.7853 - loss: 0.1123 - val_dice_coefficient: 0.7484 - val_loss: 0.1325 - learning_rate: 2.0000e-05
Epoch 8/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - dice_coefficient: 0.7952 - loss: 0.1073
Epoch 8: val_dice_coefficient did not improve from 0.74836
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 178s 169ms/step - dice_coefficient: 0.7900 - loss: 0.1100 - val_dice_coefficient: 0.7482 - val_loss: 0.1326 - learning_rate: 2.0000e-05
Epoch 9/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - dice_coefficient: 0.7835 - loss: 0.1136
Epoch 9: val_dice_coefficient did not improve from 0.74836
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 178s 169ms/step - dice_coefficient: 0.7859 - loss: 0.1120 - val_dice_coefficient: 0.7476 - val_loss: 0.1328 - learning_rate: 2.0000e-05
Epoch 10/15
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - dice_coefficient: 0.7820 - loss: 0.1141